# ML-10 — Content Action Playbook
Jackline Mutheu — Refresh / Content Opportunity Scoring

Working with an AI assistant: read `skills/README.md`, then load `writing-honest-claims` + `flyrank/flyrank-data` — done before writing this notebook.


## 1. Ranked actions + reason codes

Built from the **honest, client-grouped Random Forest model** (Week 5/6), not the naive-split version — that distinction matters after Week 6 showed how much a bad split can inflate confidence.

Four archetypes, rule-based on the model's predicted probability, the Week-4 baseline's CTR-gap flag, and freshness tier:

| Archetype | Reason code | Recommended action |
|---|---|---|
| `declining_and_ctr_gap` | Model flags likely decline **and** CTR sits below position-tier peers | **Priority review — CTR fix.** Title/snippet review first; the page is visible enough that a CTR fix has real reach. |
| `declining_and_stale` | Model flags likely decline **and** page hasn't been updated in 91+ days | **Priority review — content refresh.** Per the FlyRank paper's freshness finding, refresh strong stale content before it decays further. |
| `declining_other` | Model flags likely decline, no clear CTR or staleness driver | **Investigate.** Flagged but the "why" isn't obvious from these two signals — needs a human look, not a templated fix. |
| `ctr_gap_only` | CTR gap present, model does **not** flag decline | **Monitor / low-priority CTR review.** Worth a look, but not urgent. |
| `no_flag` | Neither signal fires | **No action.** |


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

RANDOM_SEED = 42
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_opportunity"] = ((df["trend_direction"] == "down") & (df["avg_position"] > 10)).astype(int)

tier_stats = df.groupby("position_tier").agg(total_clicks=("clicks_90d","sum"), total_impr=("impressions_90d","sum"))
tier_stats["expected_ctr_pct"] = tier_stats["total_clicks"]/tier_stats["total_impr"]*100
df["expected_ctr_pct"] = df["position_tier"].map(tier_stats["expected_ctr_pct"].to_dict())
df["ctr_gap"] = df["expected_ctr_pct"] - df["ctr"]
df["visible"] = (df["impressions_90d"] >= 300).astype(int)
df["ctr_gap_positive"] = df["ctr_gap"].clip(lower=0)
df["baseline_score"] = df["visible"]*df["ctr_gap_positive"]*df["impressions_90d"]

numeric_features = ["search_volume","competition","cpc","word_count","char_count","impressions_90d","clicks_90d",
    "pageviews_90d","sessions_90d","users_90d","engaged_sessions_90d","ai_sessions_90d","scroll_events_90d",
    "days_with_impressions","days_with_sessions","clicks_last_30d","sessions_last_30d","clicks_prev_30d",
    "sessions_prev_30d","content_age_days","days_since_last_update","ctr","engagement_rate","scroll_rate","ai_traffic_pct"]
categorical_features = ["content_type","main_intent","competition_level","freshness_tier","provider_used","model_used"]
for c in categorical_features: df[c] = df[c].fillna("missing")

# Honest split - client-grouped, matching Week 5/6
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
train_idx, val_idx = next(gss.split(df, groups=df["client_id"]))
train, val = df.iloc[train_idx].copy(), df.iloc[val_idx].copy()

preprocess = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)])
rf = Pipeline([("prep", preprocess), ("clf", RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=20, random_state=RANDOM_SEED, n_jobs=-1))])
rf.fit(train[numeric_features+categorical_features], train["is_opportunity"])
val["model_score"] = rf.predict_proba(val[numeric_features+categorical_features])[:,1]

def archetype_and_reason(row):
    high_conf = row["model_score"] >= 0.5
    ctr_flag = row["ctr_gap_positive"] > 0 and row["visible"] == 1
    stale = row["freshness_tier"] in ["91-180", "181+"]
    if high_conf and ctr_flag:
        return "declining_and_ctr_gap", "priority_review_ctr_fix"
    elif high_conf and stale:
        return "declining_and_stale", "priority_review_content_refresh"
    elif high_conf:
        return "declining_other", "investigate"
    elif ctr_flag:
        return "ctr_gap_only", "monitor_ctr"
    else:
        return "no_flag", "no_action"

val[["archetype", "reason_code"]] = val.apply(lambda r: pd.Series(archetype_and_reason(r)), axis=1)

print("Archetype counts and observed opportunity rate (for reference only, not a guarantee):")
print(val.groupby("archetype")["is_opportunity"].agg(n="size", observed_rate="mean").round(3))


Archetype counts and observed opportunity rate (for reference only, not a guarantee):
                          n  observed_rate
archetype                                 
ctr_gap_only           2338          0.280
declining_and_ctr_gap   202          0.342
declining_and_stale     140          0.286
declining_other         279          0.280
no_flag                4156          0.194


## 2. Intended use and limits

**Who uses this:** A content strategist or SEO lead deciding which pages to review first out of a large backlog — a prioritization aid, not an auto-pilot.

**What it's valid for:** Ranking pages by *likelihood of being worth a human look*, within this dataset's 90-day window, for clients resembling the ones in this training set.

**Where it stops being valid:**
- **New clients with no history yet** — the model has never seen their pattern; treat any score for a brand-new client as unverified until a few months of data exist.
- **Beyond the 90-day window** — the labels and features are all trailing 90-day aggregates; this says nothing about longer seasonal cycles.
- **As a causal claim** — per `writing-honest-claims`, this is a decision-support ranking, not a controlled experiment. A high score means "this page resembles others that were declining," not "reviewing it will fix anything."
- **The FlyRank paper's freshness finding vs. my own signal check** — the paper found a strong 31-90-day freshness window (7.88:1 growth-to-decline ratio). My own Week-4 staleness signal check came back **MIXED** — decline rate rose with staleness up to the 91-180 day tier, then *reversed* at 181+ days (n=174). I'm noting the disagreement rather than picking whichever result is more flattering: freshness matters, but not as a clean monotonic rule in this dataset.


In [2]:
print("Freshness tier vs decline rate (Week-4 signal check, restated for the playbook):")
print(df.groupby("freshness_tier")["is_opportunity"].agg(n="size", opportunity_rate="mean").round(3))


Freshness tier vs decline rate (Week-4 signal check, restated for the playbook):
                    n  opportunity_rate
freshness_tier                         
0-30            20480             0.270
181+              174             0.207
31-90             175             0.406
91-180           9171             0.361


## 3. Human review + the no-go list

**A human must confirm before any action is taken:**
- That the page's content is actually stale/weak on manual read, not just statistically resembling other declining pages.
- That a "CTR fix" archetype page's title/snippet genuinely looks like the problem, rather than a rich-result or featured snippet absorbing clicks above it.
- That a flagged page isn't intentionally low-CTR by design (e.g., a definition box meant to satisfy the search without a click).

**What should NEVER be automated:**
- **Auto-publishing any content change.** This model ranks review priority — it does not write, approve, or ship content changes on its own.
- **Auto-deprioritizing or removing a page** based on a `no_flag` result. Absence of a flag is not evidence the page is healthy; it just didn't clear this specific bar.
- **Cross-client comparisons of raw scores.** Per the Week-3 data contract, client history coverage is uneven — comparing one client's score directly against another's without accounting for that would misread the ranking.
- **Treating `declining_and_ctr_gap` pages as guaranteed wins.** The archetype's observed opportunity rate (34.2%) is higher than baseline (23.2%) but still means roughly two-thirds of flagged pages in this bucket were not, in fact, true opportunities in this data.


In [3]:
print("No computation needed - this section is written guardrails.")
print("Observed opportunity rate by archetype, restated as the 'two-thirds' figure above traces to:")
print(val.groupby('archetype')['is_opportunity'].mean().round(3))


No computation needed - this section is written guardrails.
Observed opportunity rate by archetype, restated as the 'two-thirds' figure above traces to:
archetype
ctr_gap_only             0.280
declining_and_ctr_gap    0.342
declining_and_stale      0.286
declining_other          0.280
no_flag                  0.194
Name: is_opportunity, dtype: float64


## 4. Monitoring / retrain triggers

**Signals that the recommendations have gone stale:**
- **Precision@50 on new data drifts down from the validated 0.320.** If a monthly spot-check of the top 50 flagged pages shows a meaningfully lower true-opportunity rate, the model is no longer matching current patterns.
- **A new client's pages dominate the top of the queue immediately.** Per the Week-4 baseline's own weak-pick finding, a scoring approach that rewards raw scale can let one client's large pages crowd out genuine smaller opportunities — worth checking whenever a new client is added.
- **The freshness-tier disagreement (Section 2) shifts.** If a future signal check finds the 181+ day reversal was noise rather than a real pattern, or finds a new reversal at a different tier, that's a sign the underlying content-lifecycle pattern has changed and the archetype rules should be revisited.
- **Retrain cadence:** quarterly at minimum, given the paper's own 90-day reporting window — sooner if any of the above drift signals fire.


In [4]:
print("Monitoring section is written guidance; no computation required.")


Monitoring section is written guidance; no computation required.


## 5. Exports for the paper


In [5]:
import os
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

out_cols = ["content_id", "client_id", "model_score", "baseline_score", "archetype", "reason_code",
            "freshness_tier", "position_tier", "ctr", "impressions_90d"]
ranked = val[out_cols].sort_values("model_score", ascending=False)
ranked.to_csv("work/outputs/action_playbook_queue.csv", index=False)
print("Wrote", len(ranked), "rows to work/outputs/action_playbook_queue.csv")

# Metrics JSON - the receipts the paper's numbers trace back to
import json as _json
metrics = {
    "validation_set_size": int(len(val)),
    "validation_base_rate": round(float(val["is_opportunity"].mean()), 3),
    "archetype_counts": val["archetype"].value_counts().to_dict(),
    "archetype_observed_opportunity_rate": val.groupby("archetype")["is_opportunity"].mean().round(3).to_dict(),
    "freshness_tier_opportunity_rate": df.groupby("freshness_tier")["is_opportunity"].mean().round(3).to_dict(),
    "note": "All rates are observed, in-sample, decision-support figures - not causal claims. See Section 2 for scope limits."
}
with open("work/outputs/playbook_metrics.json", "w") as f:
    _json.dump(metrics, f, indent=2)
print("Wrote work/outputs/playbook_metrics.json")

# Figure - archetype counts, for reuse in the paper
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

counts = val["archetype"].value_counts()
fig, ax = plt.subplots(figsize=(7,4))
ax.bar(counts.index, counts.values, color="#2B4C5C")
ax.set_ylabel("Pages in validation set")
ax.set_title("Content archetypes (client-grouped validation set)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig("work/figures/archetype_counts.png", dpi=150)
print("Saved work/figures/archetype_counts.png")


Wrote 7115 rows to work/outputs/action_playbook_queue.csv
Wrote work/outputs/playbook_metrics.json


Saved work/figures/archetype_counts.png


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere (client/content IDs are the repo's own pseudonyms)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit my repo URL on the card. Done.


---

## 5-Minute Demo Outline (Week-8 showcase, optional)

**Question (30 sec):** Out of thousands of content pages, which ones actually deserve a human's time to review first — and can a simple hand-written rule answer that as well as a trained model?

**Method (90 sec):** Framed as a proxy classification task (declining trend + poor position), built a hand-written CTR-gap baseline first, checked it against two real signals before trusting it, then trained Logistic Regression and Random Forest on a **client-grouped** validation split — not a random one, after Week 6 showed random splits can inflate results by leaking the same client's pages across train and test.

**One chart (60 sec):** `work/figures/archetype_counts.png` — five content archetypes (declining+CTR-gap, declining+stale, declining-other, CTR-gap-only, no-flag), showing how the ranked queue actually breaks down, not just a single score.

**One honest result (90 sec):** The hand-written baseline scored *worse than random selection* on the real target (Precision@50 of 0.040 vs. a 0.232 base rate) — it was built for a related but different question. Both trained models clearly beat it; Random Forest won on Precision@50 (0.320) despite a lower overall ROC-AUC than Logistic Regression, and I reported both numbers rather than the more flattering one.

**One recommendation (30 sec):** Use the model's ranked queue to prioritize human review — never auto-publish changes based on it — and re-check Precision@K quarterly, since a naive random split proved capable of hiding almost 0.2 points of AUC inflation.

---

## Two Shareable Cuts

**Short social post (methodology-focused):**
> Spent this internship build learning the hard way that a good baseline isn't optional — it's the thing your "better" model actually has to beat. Built a hand-written CTR-gap rule first, then found it scored *worse than random* on the real target once I evaluated it honestly. A trained model beat it, but only after I caught a data-leakage bug (two features that literally encoded the label) and switched from a random train/test split to a client-grouped one — which alone moved my validation AUC by ~0.2 points. The boring parts (the baseline, the split, the leakage check) turned out to be the actual work. #MachineLearning #DataScience

**3-sentence employer-facing summary:**
> I built a decision-support model that ranks content pages by likelihood of needing a refresh, using FlyRank's anonymized search-performance data. After validating that a hand-written baseline rule underperformed random selection on the true target, I trained and honestly evaluated Logistic Regression and Random Forest models on a client-grouped split, catching a real data-leakage issue along the way and reporting both winning and losing metrics rather than the more flattering one. The result is a prioritization tool for human reviewers — not an autonomous system — with documented limits, a no-go list, and monitoring triggers for when it should be retrained.
